In [1]:
# import os
# current_path = os.getcwd()
# print(current_path)

In [2]:
# import sys
# print(sys.executable)

Важный момент!! 

после скачивания файлов важно открыть скрытый файлик .env в корне папки и проверить ключ api openroute. сейчас там стоит мой ключ какой-то бесплатный, потом можно поменять. не знаю, какой там лимит

настраиваем окружение и импорты + пути к файлам нас интересующих

In [3]:
import os
import sys
import json
import logging
import pandas as pd
import csv  # <--- ДОБАВЛЕНО: Этого не хватало, из-за этого была ошибка
from datetime import datetime
from dotenv import load_dotenv

# Находим корневую папку проекта 
notebook_path = os.path.abspath('.')
project_root = os.path.dirname(notebook_path) 
sys.path.insert(0, project_root)

# Загружаем переменные из .env файла, который лежит в корне проекта
env_path = os.path.join(project_root, '.env')
if os.path.exists(env_path):
    load_dotenv(env_path)
    print(f"Переменные окружения загружены из: {env_path}")
else:
    print("Файл .env не найден! также убедитесь, что OPEN_ROUTER_API_KEY и MODEL_NAME заданы.")

# Настраиваем логирование, чтобы видеть прогресс
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Импорт компонентов системы
try:
    # Пайплайн и Судья
    from app.orchestrator.runner import run
    from app.judge.agent import JudgeAgent
    # Генератор и Фиксер
    from app.generator.agent import GeneratorAgent
    from app.fixer.agent import FixerAgent
    
    # Модели и утилиты
    from app.models.state import GraphState
    from app.models.schemas import Verdict, IssueType
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_openai import ChatOpenAI
    from app.config import get_config
    
    print("Все модули успешно импортированы")
except ImportError as e:
    logger.error(f"Ошибка импорта: {e}. убедитесь, что зависимости установлены через pip3.11 install -r requirements.txt")

# Пути к датасетам (строго по твоей структуре: папка data в корне)
DATA_DIR = os.path.join(project_root, 'data')
SAFE_DATASET_PATH = os.path.join(DATA_DIR, 'dataset.csv')
VULN_DATASET_PATH = os.path.join(DATA_DIR, 'vuln_dataset.csv')

Переменные окружения загружены из: /Users/sofianekrasova/testov/.env
Все модули успешно импортированы


функции загрузки данных и метрик 

In [4]:
def load_safe_dataset():
    if not os.path.exists(SAFE_DATASET_PATH):
        logger.error(f"Файл {SAFE_DATASET_PATH} не найден!")
        return []
    
    try:
        # Читаем CSV с учетом того, что поля могут быть многострочными и содержать кавычки
        df = pd.read_csv(
            SAFE_DATASET_PATH, 
            sep=';',           # Твой разделитель для dataset.csv
            quotechar='"',     # Символ кавычки
            quoting=csv.QUOTE_MINIMAL, # Используем стандартное квотирование
            engine='python',   # Движок python лучше справляется со сложными CSV и переносами строк внутри полей
            on_bad_lines='warn' # Предупреждать о битых строках, но не падать
        )
        
        # Нормализация имен колонок под твой код теста
        # В твоем файле колонки называются "text" и "sql" (судя по приложенному содержимому)
        # Если вдруг они называются иначе, раскомментируй проверку ниже:
        
        # if 'task_description' in df.columns and 'sql' in df.columns:
        #      df.rename(columns={'task_description': 'text'}, inplace=True)
             
        return df.to_dict('records')
        
    except Exception as e:
        logger.error(f"Критическая ошибка чтения safe dataset: {e}")
        # Для отладки можно вывести первые строки сырого файла, если ошибка парсинга
        return []

def load_vulnerable_dataset():
    if not os.path.exists(VULN_DATASET_PATH):
        logger.error(f"Файл {VULN_DATASET_PATH} не найден!")
        return []
    
    try:
        df = pd.read_csv(
            VULN_DATASET_PATH, 
            sep=',',           # Твой разделитель для vuln_dataset.csv (обычно запятая)
            quotechar='"',     # Символ кавычки
            quoting=csv.QUOTE_MINIMAL,
            engine='python',
            on_bad_lines='warn'
        )
        
        # Нормализация имен колонок
        # Убедись, что в vuln_dataset.csv колонки называются именно так:
        # id, task_description, vulnerable_sql, expected_vuln_class, risk_score
        
        if 'vulnerable_sql' not in df.columns and 'sql' in df.columns:
            df.rename(columns={'sql': 'vulnerable_sql'}, inplace=True)
        if 'expected_vuln_class' not in df.columns and 'vuln_type' in df.columns:
            df.rename(columns={'vuln_type': 'expected_vuln_class'}, inplace=True)
            
        return df.to_dict('records')
        
    except Exception as e:
        logger.error(f"Критическая ошибка чтения vuln dataset: {e}")
        return []

def calculate_execution_accuracy(results):
    if not results: return 0.0
    approved_count = sum(1 for r in results if r.get('approved', False))
    return round((approved_count / len(results)) * 100, 2)

def calculate_judge_coverage(judge_results):
    total_tests = len(judge_results)
    detected_correctly = 0
    missed_vulns = []
    found_vuln_types = set()

    for res in judge_results:
        expected_vuln_class = res.get('expected_vuln_class')
        found_issues = res.get('found_issues', [])
        
        has_critical_issue = any(i.severity.value in ['HIGH', 'MEDIUM'] for i in found_issues)
        is_rejected = res.get('verdict') == Verdict.REJECTED
        
        for issue in found_issues:
            found_vuln_types.add(issue.type.value)

        if has_critical_issue or is_rejected:
            detected_correctly += 1
        else:
            missed_vulns.append({
                "id": res.get('id'),
                "expected_vuln": expected_vuln_class,
                "verdict": res.get('verdict'),
                "issues_found": [i.type.value for i in found_issues]
            })
            
    coverage = (detected_correctly / total_tests) * 100 if total_tests > 0 else 0
    
    return {
        "coverage_percent": round(coverage, 2),
        "total_tests": total_tests,
        "detected": detected_correctly,
        "unique_vuln_types_found": list(found_vuln_types),
        "missed_cases": missed_vulns
    }

def analyze_iterations(pipeline_results):
    if not pipeline_results: return {}
    iterations_list = [r['iterations_used'] for r in pipeline_results]
    avg_iterations = sum(iterations_list) / len(iterations_list)
    
    risk_reduction_success = 0
    total_risk_start = 0
    total_risk_end = 0

    for r in pipeline_results:
        logs = r.get('iteration_logs', [])
        if logs:
            initial_risk = logs[0].judge_output.risk_score
            final_risk = logs[-1].judge_output.risk_score
            total_risk_start += initial_risk
            total_risk_end += final_risk
            if final_risk < initial_risk or (final_risk == 0 and r['approved']):
                risk_reduction_success += 1
                
    reduction_rate = (risk_reduction_success / len(pipeline_results)) * 100 if pipeline_results else 0
    avg_risk_start = total_risk_start / len(pipeline_results)
    avg_risk_end = total_risk_end / len(pipeline_results)

    return {
        "avg_iterations": round(avg_iterations, 2),
        "max_iterations": max(iterations_list) if iterations_list else 0,
        "min_iterations": min(iterations_list) if iterations_list else 0,
        "risk_reduction_rate": round(reduction_rate, 2),
        "avg_risk_start": round(avg_risk_start, 2),
        "avg_risk_end": round(avg_risk_end, 2)
    }

функции тестирования пайплайна, генератора, фиксера и судьи

In [5]:
def calculate_generator_metrics(gen_results):
    """Оценивает качество работы Генератора"""
    if not gen_results: return {}
    total = len(gen_results)
    valid_sql_count = sum(1 for r in gen_results if r.get('is_valid_json', False))
    # Можно добавить проверку на наличие LIMIT или отсутствие SELECT *, если есть эталон
    return {
        "total_tasks": total,
        "valid_json_responses": valid_sql_count,
        "success_rate": round((valid_sql_count / total) * 100, 2) if total > 0 else 0
    }

def calculate_fixer_metrics(fixer_results):
    """Оценивает качество работы Фиксера"""
    if not fixer_results: return {}
    total = len(fixer_results)
    fixed_count = sum(1 for r in fixer_results if r.get('is_fixed', False))
    return {
        "total_tasks": total,
        "successfully_fixed": fixed_count,
        "fix_rate": round((fixed_count / total) * 100, 2) if total > 0 else 0
    }
    
def test_full_pipeline(safe_tasks):
    logger.info("Запуск теста полного пайплайна (Safe Tasks)")
    results = []
    # Берем первые 50 задач для скорости, или все если нужно
    tasks_to_test = safe_tasks[:50] 
    
    for task_data in tasks_to_test:
        task_desc = task_data.get('text', '') or task_data.get('task_description', '')
        if not task_desc or len(str(task_desc)) < 5:
            continue

        logger.info(f"Обработка задачи: {str(task_desc)[:40]}...")
        try:
            # Вызываем основной раннер системы
            result = run(str(task_desc))
            results.append({
                "task_preview": str(task_desc)[:50],
                "final_sql": result.final_sql,
                "approved": result.approved,
                "iterations": result.iterations_used,
                "logs": result.iteration_logs
            })
        except Exception as e:
            logger.error(f"Ошибка при выполнении задачи: {e}")
            results.append({
                "task_preview": str(task_desc)[:50],
                "error": str(e),
                "approved": False,
                "iterations": 0
            })
    return results

def test_judge_agent(vuln_tasks):
    logger.info("=== Запуск теста агента-судьи (Vuln Tasks) ===")
    results = []
    
    # Получаем конфигурацию для создания LLM
    config = get_config()
    
    # Важно: проверяем, что модель указана в конфиге/переменных окружения
    if not config.model_name:
        raise ValueError("MODEL_NAME не найдена в .env файле или config.py")

    llm = ChatOpenAI(
        base_url=config.openrouter_base_url,
        api_key=config.open_router_api_key,
        model=config.model_name
    )
    
    judge = JudgeAgent(llm=llm)
    
    for vuln_data in vuln_tasks:
        vuln_sql = vuln_data.get('vulnerable_sql', '')
        expected_vuln = vuln_data.get('expected_vuln_class', '')
        
        if not vuln_sql:
            continue

        logger.info(f"Проверка уязвимости [{expected_vuln}]: {vuln_sql[:30]}...")
        
        state = {
            "messages": [HumanMessage(content=f"Проверь этот SQL на безопасность: {vuln_sql}")],
            "judge_responses": [],
            "llm_calls": 0,
            "audit_iteration": 0
        }
        
        try:
            output = judge(state)
            last_response = output['judge_responses'][-1]
            
            results.append({
                "id": vuln_data.get('id'),
                "sql_preview": vuln_sql[:50],
                "expected_vuln_class": expected_vuln,
                "verdict": last_response.verdict,
                "risk_score": last_response.risk_score,
                "found_issues": last_response.issues
            })
        except Exception as e:
            logger.error(f"Ошибка судьи на задаче: {e}")
            
    return results

def test_generator_agent(safe_tasks):
    """Тестирует Генератор изолированно: Текст -> SQL"""
    logger.info("Запуск теста Генератора")
    results = []
    config = get_config()
    llm = ChatOpenAI(
        base_url=config.openrouter_base_url,
        api_key=config.open_router_api_key,
        model=config.model_name
    )
    generator = GeneratorAgent(llm=llm)
    
    # Берем немного задач для быстрого теста
    tasks_to_test = safe_tasks[:20]
    for task_data in tasks_to_test:
        task_desc = task_data.get('text', '')
        if not task_desc: continue
        
        logger.info(f"Генерация для: {task_desc[:30]}...")
        state = {
            "messages": [HumanMessage(content=task_desc)],
            "llm_calls": 0
        }
        try:
            output = generator(state)
            # Парсим ответ генератора
            ai_msg = output['messages'][0]
            content = json.loads(ai_msg.content)
            sql = content.get('sql', '')
            tables = content.get('tables_used', [])
            
            results.append({
                "task": task_desc[:50],
                "generated_sql": sql,
                "tables_used": tables,
                "is_valid_json": True
            })
        except Exception as e:
            logger.error(f"Ошибка генератора: {e}")
            results.append({
                "task": task_desc[:50],
                "error": str(e),
                "is_valid_json": False
            })
    return results

def test_fixer_agent(vuln_tasks):
    """Тестирует Фиксер изолированно: Плохой SQL + Ошибки -> Хороший SQL"""
    logger.info("Запуск теста Фиксера")
    results = []
    config = get_config()
    llm = ChatOpenAI(
        base_url=config.openrouter_base_url,
        api_key=config.open_router_api_key,
        model=config.model_name
    )
    fixer = FixerAgent(llm=llm)
    
    # Берем задачи с уязвимостями
    tasks_to_test = vuln_tasks[:20]
    
    for vuln_data in tasks_to_test:
        bad_sql = vuln_data.get('vulnerable_sql', '')
        vuln_type = vuln_data.get('expected_vuln_class', 'UNKNOWN')
        if not bad_sql: continue
        
        logger.info(f"Исправление уязвимости [{vuln_type}]...")
        
        # Формируем фейковые issues для фиксера
        fake_issues = [
            {
                "type": "SECURITY",
                "severity": "HIGH",
                "message": f"Detected vulnerability: {vuln_type}",
                "fix_instruction": "Fix the SQL query to remove the vulnerability while preserving logic."
            }
        ]
        
        # Создаем состояние для фиксера
        # Фиксеру нужно сообщение с SQL и judge_responses
        state = {
            "messages": [
                HumanMessage(content="Generate SQL"), # Фейк, чтобы было что парсить
                SystemMessage(content=json.dumps({"sql": bad_sql, "tables_used": ["dummy"]})) # Фейк генератора
            ],
            "judge_responses": [{
                "verdict": Verdict.REJECTED,
                "risk_score": 9.0,
                "issues": fake_issues
            }],
            "llm_calls": 0
        }
        
        try:
            output = fixer(state)
            ai_msg = output['messages'][0]
            content = json.loads(ai_msg.content)
            fixed_sql = content.get('sql', '')
            
            # Простая проверка: изменился ли SQL?
            is_fixed = (fixed_sql != bad_sql) and len(fixed_sql) > 0
            
            results.append({
                "original_sql": bad_sql[:50],
                "fixed_sql": fixed_sql[:50],
                "vuln_type": vuln_type,
                "is_fixed": is_fixed
            })
        except Exception as e:
            logger.error(f"Ошибка фиксера: {e}")
            results.append({
                "original_sql": bad_sql[:50],
                "error": str(e),
                "is_fixed": False
            })
    return results

а теперь главный запуск и вывод результатов

In [6]:
print("Начало комплексного тестирования системы secure-sql-agents")
print("-" * 60)

# 1. Загрузка данных
safe_tasks = load_safe_dataset()
vuln_tasks = load_vulnerable_dataset()

all_reports = {}

    # 2. Тестирование
#весь пайплайн
pipeline_results = []
    
if safe_tasks:
    pipeline_results = test_full_pipeline(safe_tasks)
    all_reports['pipeline'] = {
        "execution_accuracy": calculate_execution_accuracy(pipeline_results),
        "iteration_stats": analyze_iterations(pipeline_results)
    }

#судья
judge_results = []
if vuln_tasks:
    judge_results = test_judge_agent(vuln_tasks)
    all_reports['judge'] = calculate_judge_coverage(judge_results)

#генератор
gen_results = []
if safe_tasks:
    gen_results = test_generator_agent(safe_tasks)
    all_reports['generator'] = calculate_generator_metrics(gen_results)

#фиксер
fixer_results = []
if vuln_tasks:
    fixer_results = test_fixer_agent(vuln_tasks)
    all_reports['fixer'] = calculate_fixer_metrics(fixer_results)

# 4. Красивый вывод в Notebook
print("ИТОГОВЫЙ ОТЧЕТ ПО ТЕСТИРОВАНИЮ")
    
if 'pipeline' in all_reports:
    p = all_reports['pipeline']
    print(f"\n1. весь пайплайн")
    print(f"   Execution Accuracy: {p['execution_accuracy']}% {'пройдено' if p['execution_accuracy'] >= 70 else 'не пройдено'}")
    print(f"   Avg Iterations: {p['iteration_stats'].get('avg_iterations', 'N/A')}")
    print(f"   Risk Reduction: {p['iteration_stats'].get('risk_reduction_rate', 'N/A')}%")

if 'judge' in all_reports:
    j = all_reports['judge']
    print(f"\n2. СУДЬЯ (Security Audit)")
    print(f"   Vulnerability Coverage: {j['coverage_percent']}%")
    print(f"   Detected: {j['detected']} / {j['total_tests']}")
    print(f"   Missed Types: {len(j['missed_cases'])}")

if 'generator' in all_reports:
    g = all_reports['generator']
    print(f"\n3. ГЕНЕРАТОР (Text-to-SQL)")
    print(f"   Valid JSON Responses: {g['valid_json_responses']} / {g['total_tasks']}")
    print(f"   Success Rate: {g['success_rate']}%")

if 'fixer' in all_reports:
    f = all_reports['fixer']
    print(f"\n4. ФИКСЕР (SQL Correction)")
    print(f"   Successfully Fixed: {f['successfully_fixed']} / {f['total_tasks']}")
    print(f"   Fix Rate: {f['fix_rate']}%")

# 4. Сохранение отчета
reports_dir = os.path.join(project_root, 'qa', 'reports')
os.makedirs(reports_dir, exist_ok=True)
report_filename = os.path.join(reports_dir, f"full_test_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")

with open(report_filename, 'w', encoding='utf-8') as f:
    json.dump(all_reports, f, indent=2, ensure_ascii=False, default=str)

print(f"\n Подробный отчет сохранен в: `{report_filename}`")

/var/folders/sb/9dfcgtd91x95qmjld9r5yz340000gn/T/ipykernel_64995/2991752768.py:8: ParserWarning: Skipping line 2: ';' expected after '"'

  df = pd.read_csv(
/var/folders/sb/9dfcgtd91x95qmjld9r5yz340000gn/T/ipykernel_64995/2991752768.py:8: ParserWarning: Skipping line 3: ';' expected after '"'

  df = pd.read_csv(
/var/folders/sb/9dfcgtd91x95qmjld9r5yz340000gn/T/ipykernel_64995/2991752768.py:8: ParserWarning: Skipping line 4: ';' expected after '"'

  df = pd.read_csv(
/var/folders/sb/9dfcgtd91x95qmjld9r5yz340000gn/T/ipykernel_64995/2991752768.py:8: ParserWarning: Skipping line 5: ';' expected after '"'

  df = pd.read_csv(
/var/folders/sb/9dfcgtd91x95qmjld9r5yz340000gn/T/ipykernel_64995/2991752768.py:8: ParserWarning: Skipping line 6: ';' expected after '"'

  df = pd.read_csv(
/var/folders/sb/9dfcgtd91x95qmjld9r5yz340000gn/T/ipykernel_64995/2991752768.py:8: ParserWarning: Skipping line 7: ';' expected after '"'

  df = pd.read_csv(
/var/folders/sb/9dfcgtd91x95qmjld9r5yz340000gn/T/ipy

Начало комплексного тестирования системы secure-sql-agents
------------------------------------------------------------


2026-05-16 03:12:44,460 - INFO - Проверка уязвимости [SQL_INJ_CLASSIC]: SELECT * FROM logs WHERE msg =...
2026-05-16 03:12:45,360 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-16 03:12:46,310 - INFO - Проверка уязвимости [SQL_INJ_CLASSIC]: SELECT * FROM orders WHERE sta...
2026-05-16 03:12:46,794 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-16 03:12:47,719 - INFO - Проверка уязвимости [SQL_INJ_CLASSIC]: EXECUTE 'SELECT * FROM users W...
2026-05-16 03:12:48,226 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-16 03:12:49,866 - INFO - Проверка уязвимости [SQL_INJ_CLASSIC]: SELECT * FROM products WHERE n...
2026-05-16 03:12:50,378 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-16 03:12:51,096 - INFO - Проверка уязвимости [SQL_INJ_CLASSIC]: SELECT * FROM customer WHERE l...
2026-0

ИТОГОВЫЙ ОТЧЕТ ПО ТЕСТИРОВАНИЮ

1. весь пайплайн
   Execution Accuracy: 0.0% не пройдено
   Avg Iterations: N/A
   Risk Reduction: N/A%

2. СУДЬЯ (Security Audit)
   Vulnerability Coverage: 98.15%
   Detected: 53 / 54
   Missed Types: 1

3. ГЕНЕРАТОР (Text-to-SQL)


KeyError: 'valid_json_responses'